# Week 8: finish and train a small GPT
Run cells in order. First choose **Runtime > Change runtime type > GPU > T4 or A100** and connect. Paid Colab does not guarantee a GPU type. Training uses your compute units; inspect the benchmark before starting. Outputs and checkpoints are saved only under the new project folder you authorize in Google Drive.
The model code was written independently. This notebook runs the repository scripts, not a copied GPT file. The six-stage CPU comparison is already recorded; large English/Turkish GPU training is still to be performed.
[Colab resource policy](https://research.google.com/colaboratory/faq.html)

## 1. Verify the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
GPU_NAME = torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME)
print('PyTorch:', torch.__version__)
print('GPU memory (GB):', torch.cuda.get_device_properties(0).total_memory / 1e9)
print('BF16 supported:', torch.cuda.is_bf16_supported())

## 2. Load the code and dependencies
Do not upgrade/reinstall CUDA PyTorch unnecessarily; Colab normally includes it. The requirements use minimum versions, not `--upgrade`. A reused checkout is preserved.

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO = Path('/content/neuron-to-gpt')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/muend/neuron-to-gpt.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'week08/requirements.txt'], check=True)

## 3. Mount Drive and choose the run
Approve the Google Drive prompt yourself. Mounting gives notebook code access to Drive; these scripts write only inside `OUTPUT`. Choose a fresh RUN_NAME for a new experiment. Keep it unchanged to resume the same run. Default: A100 preset on an A100, otherwise the smaller T4 preset. You may select `t4` even on an A100.
The same PRESET, STEPS and BATCH_SIZE must be used for both languages. If the initial benchmark runs out of memory, lower BATCH_SIZE to 32 before training (and benchmark that exact batch using the CLI flag in the next cell). Do not change critical settings midway through a saved run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PRESET = 'a100' if 'A100' in GPU_NAME else 't4'
STEPS = 5000
BATCH_SIZE = 64
RUN_NAME = f'week08_{PRESET}_01'
OUTPUT = Path('/content/drive/MyDrive/neuron-to-gpt') / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Preset:', PRESET, '| steps per language:', STEPS, '| batch:', BATCH_SIZE)
print('Checkpoint folder:', OUTPUT)

## 4. Prepare Shakespeare and run tests
Turkish data must be the user-selected Nazim Hikmet collection; no Turkish full texts are bundled or downloaded automatically. English training can proceed before Turkish sources are supplied. Copy exact inputs into the run's Drive folder for restart-safe training.


In [ ]:
import shutil
LOCAL_DATA = REPO / 'week08/data'
LOCAL_DATA.mkdir(exist_ok=True)
BACKUP = OUTPUT / 'corpus'
BACKUP.mkdir(exist_ok=True)
if (BACKUP / 'shakespeare.txt').exists():
    shutil.copy2(BACKUP / 'shakespeare.txt', LOCAL_DATA / 'shakespeare.txt')
subprocess.run([sys.executable, 'week08/prepare_data.py', '--english-only'], check=True)
if not (BACKUP / 'shakespeare.txt').exists():
    shutil.copy2(LOCAL_DATA / 'shakespeare.txt', BACKUP / 'shakespeare.txt')
subprocess.run([sys.executable, 'week08/test_gpt.py'], check=True)

## 5. Measure speed before starting
Ten warmup steps and fifty measured steps estimate training-only time for ONE language. Add evaluation/checkpoint overhead, and double the training-only estimate for two languages. The probe does not save a trained model. Use the measured time and your remaining compute units to decide whether to continue or select the smaller preset.

In [ ]:
subprocess.run([sys.executable, 'week08/benchmark.py', '--preset', PRESET,
                '--batch-size', str(BATCH_SIZE), '--device', 'cuda'], check=True)

## 6. Read the six-stage comparison; optional GPU rerun
The existing CPU measurements use 4,000 updates for every stage, not smoke runs. GPU rerunning them is optional and costs additional compute. If enabled, completed stages are resumed rather than discarded. Small models can be faster on CPU than GPU because kernel-launch overhead matters.

In [ ]:
import pandas as pd
from IPython.display import display
display(pd.read_csv('week08/results/cpu_ablations.csv'))
RERUN_ABLATIONS = False
if RERUN_ABLATIONS:
    subprocess.run([sys.executable, 'week08/run_ablations.py', '--device', 'cuda',
                    '--output-root', str(OUTPUT), '--resume'], check=True)

## 7. Train Shakespeare
This cell starts the full GPU run. FP16 + GradScaler is used on T4; BF16 is used where supported, normally A100. Validation is FP32. Every 500 steps saves `last.pt` (resume state) and, when improved, `best.pt` (best sampled validation checkpoint). Step count is TOTAL desired steps, not additional steps on resume.
After a disconnect, reconnect to the same GPU type, rerun cells 1-4, then this cell. A checkpoint must keep the same model, batch size, learning rate, precision and corpus. Checksum mismatches stop the run rather than silently continuing.

In [ ]:
def train_language(language):
    command = [sys.executable, '-u', 'week08/train.py', '--language', language,
               '--preset', PRESET, '--device', 'cuda', '--steps', str(STEPS),
               '--batch-size', str(BATCH_SIZE), '--output-root', str(OUTPUT)]
    if (OUTPUT / f'{language}_{PRESET}_layernorm' / 'last.pt').exists():
        command.append('--resume')
    subprocess.run(command, check=True)
train_language('en')

## 8. Prepare Nazim Hikmet sources and train Turkish
Upload the private `nazim_sources_for_colab.zip` to `MyDrive/neuron-to-gpt`, or upload the selected TXT files and `sources.json` to its `nazim_sources` subfolder. The source package contains the owner's PDF-derived collection, not GitHub-hosted full texts. Raw inputs remain private. Corpus size and OCR limitations are recorded; no repetition is used to pad the data.
This cell stops if sources are missing or a saved corpus is not the selected Nazim collection. It trains a separately initialized model with the same architecture, not the English checkpoint. Do not directly rank languages by raw NLL.

In [ ]:
sys.path.insert(0, str(REPO / 'week08'))
from data import validate_turkish_corpus
SOURCE_DIR = Path('/content/drive/MyDrive/neuron-to-gpt/nazim_sources')
SOURCE_ZIP = Path('/content/drive/MyDrive/neuron-to-gpt/nazim_sources_for_colab.zip')
if not (SOURCE_DIR / 'sources.json').exists() and SOURCE_ZIP.exists():
    import zipfile
    SOURCE_DIR = Path('/content/nazim_sources')
    SOURCE_DIR.mkdir(exist_ok=True)
    allowed = {'poems.txt', 'stories.txt', 'aydinlik_prose.txt', 'sources.json', 'quality_report.json', 'overlap_audit.json'}
    with zipfile.ZipFile(SOURCE_ZIP) as package:
        assert set(package.namelist()).issubset(allowed), 'Unexpected source package entries'
        for name in package.namelist():
            destination = SOURCE_DIR / name
            raw = package.read(name)
            if destination.exists():
                assert destination.read_bytes() == raw, 'Existing local source differs; review it first'
            else:
                destination.write_bytes(raw)
if (BACKUP / 'turkish.txt').exists() or (BACKUP / 'corpus_manifest.json').exists():
    validate_turkish_corpus(BACKUP)
    for name in ('turkish.txt', 'corpus_manifest.json'):
        shutil.copy2(BACKUP / name, LOCAL_DATA / name)
    subprocess.run([sys.executable, 'week08/prepare_data.py'], check=True)
else:
    subprocess.run([sys.executable, 'week08/prepare_data.py', '--source-dir', str(SOURCE_DIR)], check=True)
    for name in ('turkish.txt', 'corpus_manifest.json'):
        shutil.copy2(LOCAL_DATA / name, BACKUP / name)
validate_turkish_corpus(LOCAL_DATA)
train_language('tr')

## 9. Inspect and download the results
The ZIP below contains metrics, plots, samples and attribution, but excludes large `.pt` checkpoints and raw corpora. Checkpoints remain in Drive. Send this ZIP back in the project chat so the measured GPU results can be reviewed and added to the repository. No fabricated GPU results are prefilled. After verifying the files, disconnect the GPU runtime to stop consuming resources.

In [ ]:
subprocess.run([sys.executable, 'week08/summarize_results.py', '--output-root', str(OUTPUT)], check=True)
from IPython.display import Markdown
display(Markdown((OUTPUT / 'submission_report.md').read_text(encoding='utf-8')))
import zipfile
ARCHIVE = Path('/content/week08_gpu_results.zip')
with zipfile.ZipFile(ARCHIVE, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT.rglob('*')):
        if path.is_file() and path.suffix in {'.json', '.csv', '.md', '.png', '.txt'} and 'corpus' not in path.relative_to(OUTPUT).parts:
            archive.write(path, path.relative_to(OUTPUT))
    if (BACKUP / 'corpus_manifest.json').exists():
        archive.write(BACKUP / 'corpus_manifest.json', 'corpus_manifest.json')
from google.colab import files
files.download(str(ARCHIVE))